In [5]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("TensorFlow version:", tf.__version__)

tf.random.set_seed(42)
np.random.seed(42)

TensorFlow version: 2.20.0


In [6]:
BASE_DIR = '/Users/emiryscn/repos/multi-class-skin-lesion-detection'
TRAIN_DIR = os.path.join(BASE_DIR, 'data/processed/train')
VAL_DIR   = os.path.join(BASE_DIR, 'data/processed/val')
TEST_DIR  = os.path.join(BASE_DIR, 'data/processed/test')

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
NUM_CLASSES = 7
SEED        = 42

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
CLASS_NAMES = [c for c in CLASS_NAMES if not c.startswith('.')]  # skip .DS_Store
print("Detected classes:", CLASS_NAMES)

READABLE_NAMES = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions'
}

Detected classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [7]:
# We pass these weights into model.fit(class_weight=...)
# so the model doesnt ignore rare classes.
# If a class has fewer samples (df, vasc, mel),
# mistakes on those classes will count more during training.

def compute_class_weights(train_dir, class_names):
    """
    Calculate class weights based on how many samples each class has.
    Fewer samples, higher weight.
    This helps balance the dataset during training.
    """

    # Count how many images are in each class folder
    counts = {}
    for cls in class_names:
        cls_path = os.path.join(train_dir, cls)
        n = len([f for f in os.listdir(cls_path) if not f.startswith('.')])
        counts[cls] = n
    
    total = sum(counts.values())
    n_classes = len(class_names)
    
    # Create a dictionary where:
    # weight = total_samples / (number_of_classes * samples_in_this_class)
    # The keys must match the numeric labels used by the model.
    class_weight = {}
    for i, cls in enumerate(class_names):
        class_weight[i] = total / (n_classes * counts[cls])

    print(f"{'Class':<8} {'Count':>6} {'Weight':>8}")
    print("-" * 24)
    for i, cls in enumerate(class_names):
        print(f"{cls:<8} {counts[cls]:>6} {class_weight[i]:>8.3f}")
    
    return class_weight

class_weights = compute_class_weights(TRAIN_DIR, CLASS_NAMES)

Class     Count   Weight
------------------------
akiec       261    4.385
bcc         412    2.778
bkl         879    1.302
df           91   12.578
mel         891    1.285
nv         5364    0.213
vasc        114   10.040


In [8]:
# This is the default loader used by all experiments.
# Augmentation is applied separately on top of this.

def load_base_dataset(directory, shuffle=True):
    """
    Load images from class-folder structure.
    Returns tf.data.Dataset of (images, one_hot_labels), pixels in [0, 1].
    NOT batched yet — batching happens after augmentation is applied.
    """
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=None,         # Return individual samples (unbatched)
        label_mode='categorical', # One-hot encoding for 7 classes
        shuffle=shuffle,
        seed=SEED,
        class_names=CLASS_NAMES   # Force consistent class ordering
    )
    
    # Normalize pixels from [0, 255] to [0, 1]
    ds = ds.map(lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
                num_parallel_calls=tf.data.AUTOTUNE)
    
    return ds

# Load raw unbatched datasets
raw_train = load_base_dataset(TRAIN_DIR, shuffle=True)
raw_val   = load_base_dataset(VAL_DIR, shuffle=False)
raw_test  = load_base_dataset(TEST_DIR, shuffle=False)

# Quick check
for img, label in raw_train.take(1):
    print(f"Image shape: {img.shape}, dtype: {img.dtype}")
    print(f"Label shape: {label.shape}, label: {label.numpy()}")
    print(f"Pixel range: [{img.numpy().min():.3f}, {img.numpy().max():.3f}]")

Found 8012 files belonging to 7 classes.
Found 1001 files belonging to 7 classes.
Found 1002 files belonging to 7 classes.
Image shape: (224, 224, 3), dtype: <dtype: 'float32'>
Label shape: (7,), label: [0. 0. 0. 0. 0. 1. 0.]
Pixel range: [0.073, 0.754]


2026-03-03 19:49:15.065489: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
